**Bioseñales y sistemas: Proyecto 1**

Daniel Lozano<br>
Samuel Ochoa<br>
Alejandro Restrepo

Este primer proyecto se centra en la utilización de herramientas de Python para la manipulación y análisis de señales de Electroencefalografía (EEG) adquiridas por repositorios libres encontrados en bases de datos en internet como OpenNeuro. En el caso de este proyecto, se hace uso del Dataset "EEG Motor Movement/Imagery Dataset", originalmente creado por Gerwin Schalk y su equipo del programa de BCI R&D, Wadsworth Center, New York State Depaartment of Health, Albany, NY para la base de datos PhysioBank.

Aquí se busca evaluar la factibilidad de construir interfaces Cerebro-Computador (BCI) no invasivas mediante un conjunto de datos que registra la actividad cerebral en condiciones de movimiento real, imaginación del movimiento y estado de reposo. En base al contenido aprendido en el curso, se asume que el enfoque principal será realizar las comparaciones e inferencias estadisticas pertinentes sobre la Densidad Espectral de Potencia (PSD) en ritmos cerebrales específicos para diferenciar las condiciones. Esta última, no solo por ser tema central de la última unidad, sino porque se puede aprovechar sus aplicaciones para identificar periodicidades escondidas en una función de variable discreta, estimar la entropía de un proceso aleatorio o porque puede proporcionar información muy valiosa sobre la dinámica interna de muchos sistemas físicos [1].

Para hablar de la BCI, primero tenemos que entender el nacimiento de las neurociencias. En 1924, el neurólogo alemán Hans Berger, siguiendo el trabajo de Richard Caton (1875), descubre la actividad eléctrica presente en el cerebro, registrando la primera actividad mediante EEG. Mediante el análisis de este, consiguió clasificar los tipos de ondas cerebrales. No fue sino hasta 1973 que Jacques Vidal introduce el término BCI en la literatura científica mientras estaba en la Universidad de California, Los Ángeles (UCLA) planteando el uso de señales EEG para el control de dispositivos externos [2].

Hasta la fecha, se han visto avances sobre la idea de Vidal, principalmente con fines médicos, donde se ha buscado implantar prótesis neuronales para la recuperación de la visión, audición o movilidad. A día de hoy la mayor expectativa se encuentra en la empresa Neuralink, dado que ha sido líder indiscutible en el desarrollo de tecnología que permita la comunicación bidireccional entre dispositivos y el cerebro.

Los conceptos de coherencia y conectividad cerebral, en su conjunto, establecen cómo las redes neuronales se comunican para procesar información. La conectividad cerebral describe las redes de conexiones funcionales y anatómicas en todo el cerebro; la comunicación a través de dicha red depende de las oscilaciones neuronales. Por otro lado, la coherencia trata de un método matemático que permite determinar si dos o más sensores, o regiones cerebrales, presentan una actividad oscilatoria neuronal similar entre sí [3].

**Metodología**

Debido a que tratamos de definir la viabilidad de formar un BCI a partir de un Dataset que compara movimientos efectuados y movimientos imaginados, será pertinente realizar un análisis de los eletrodos más significativos a la región motora (eletrodos C4, C3 y Cz) respecto a las bandas de frecuencias Mu y Beta, que reflejan cambios durante la imaginación o ejecución de movimiento [4].

In [ ]:
import mne
import re
import numpy as np
import os
import matplotlib.pyplot as plt
import scipy.signal as signal

In [ ]:
#Codigo para el filtrado de la señal
def filtro_MuBeta(raw):
    """La función recibe el archivo .raw y retorna las 2 señales filtradas, por cada banda.
    Mu (8-13 Hz) y Beta (14-30 Hz).
    """
    raw_mu = raw.copy().filter(
        l_freq = 8.0, h_freq = 12.0,
        method='fir', fir_window='hamming', phase='zero', verbose='False'
    )
    raw_beta = raw.copy().filter(
        l_freq = 13.0, h_freq = 30.0,
        method='fir', fir_window='hamming', phase='zero', verbose='False'
    )
    return raw_mu, raw_beta

In [ ]:
#Código para la carga y filtrado de los archivos
carpeta_datos = './sujetos_eeg/'
sujetos = ['001', '002', '003']
runs = ['1', '2', '3', '4']
canales = ['C3', 'C4', 'Cz']

#Cada entrada es una fila por sujeto [C3, C4, Cz]
registros_mu_izq, registros_mu_der = [], []
registros_beta_izq, registros_beta_der = [], []

for s in sujetos:
    epocas_mu_sujeto = []
    epocas_beta_sujeto = []

    for r in runs:
        nombre_base = f"sub-{s}_task-motion_run-{r}_eeg"
        archivo_set = os.path.join(carpeta_datos, f'{nombre_base}.set')

        if not os.path.exists(archivo_set):
            continue
        
        raw = mne.io.read_raw_eeglab(archivo_set, preload=True, verbose=False)
        raw.set_eeg_reference('average', projection=False, verbose=False)
        raw.pick(canales) #Para solo quedarnos con los canales C3, C4 y Cz

        raw_mu, raw_beta = filtro_MuBeta(raw)

        events, _ = mne.events_from_annotations(raw, verbose=False)

**Referencias**

[1] D. G. Manolakis y V. K. Ingle, Tratamiento digital de señales: principios, algoritmos y aplicaciones. México: Prentice Hall, 1997.<br>
[2] J. J. Vidal, “Toward direct brain–computer communication,” Annual Review of Biophysics and Bioengineering, vol. 2, pp. 157–180, 1973, doi: 10.1146/annurev.bb.02.060173.001105.<br>
[3] S. M. Bowyer, “Coherence: A measure of the brain networks: past and present,” Neuropsychiatric Electrophysiology, vol. 2, no. 1, 2016, doi: 10.1186/s40810-015-0015-7.<br>
[4] H. Yu, S. Ba, Y. Guo, L. Guo, y G. Xu, “Effects of motor imagery tasks on brain functional networks based on EEG mu/beta rhythm,” Brain Sciences, vol. 12, no. 2, p. 194, 2022, doi: 10.3390/brainsci12020194.